# Lab 09 Challenge: worked solution

For the instructor. This notebook starts on its own: it rebuilds the Ops API, the write tool with Part E's key, and
the copilot with approval. Each task runs on its own threads.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "JUDGE_MODEL", "EMBED_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
JUDGE_MODEL       gpt-4o-mini
EMBED_MODEL       text-embedding-3-small
OPENAI_API_KEY    found in .env


The pieces from the lab: the Ops API, `show_rows`, the write tool and its key, the copilot.

In [3]:
import json
import sqlite3
import uuid
from typing import Literal

import httpx
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from shared import ops_tools
from shared.copilot import search_policies
from shared.ops_api import OPS_DB, start_ops_api
from shared.ops_tools import API, RepContext
from shared.settings import chat_model

server = start_ops_api()
http = httpx.Client(timeout=5.0)

def show_rows():
    db = sqlite3.connect(f"file:{OPS_DB}?mode=ro", uri=True)
    for row in db.execute("SELECT request_id, invoice_id, amount, status, idempotency_key FROM refund_requests"):
        print(row)
    db.close()

def idempotency_key(runtime, invoice_id, amount):
    thread_id = runtime.config["configurable"]["thread_id"]
    return f"{thread_id}:{invoice_id}:{amount:.2f}"

@tool(parse_docstring=True)
def file_refund_request(invoice_id: int, amount: float,
                        reason: Literal["duplicate_charge", "damaged_media", "other"],
                        runtime: ToolRuntime[RepContext]) -> str:
    """File a refund request. A person reviews it first.

    Args:
        invoice_id: The invoice to refund, digits only.
        amount: US dollars, at most the invoice total.
        reason: Why the refund is due.
    """
    body = {"invoice_id": invoice_id, "amount": amount, "reason": reason, "rep_id": runtime.context.rep_id}
    key = idempotency_key(runtime, invoice_id, amount)
    response = http.post(f"{API}/refund-requests", json=body, headers={"Idempotency-Key": key})
    if response.status_code in (404, 422):
        return "Not filed: " + response.json()["detail"]
    response.raise_for_status()
    return response.text

Ops API started at http://127.0.0.1:8765


In [4]:
PROMPT = ("You help Chinook support reps.\n"
          "Before you file a refund, read each invoice with get_invoice and check the rule with search_policies.\n"
          "A filed refund request waits for a person to review it. Never say that money has been sent.")
TOOLS = [search_policies, ops_tools.get_invoice, file_refund_request]
approval = HumanInTheLoopMiddleware(interrupt_on={
    "file_refund_request": {"allowed_decisions": ["approve", "edit", "reject"]}})
agent = create_agent(chat_model(settings), tools=TOOLS, system_prompt=PROMPT, middleware=[approval],
                     checkpointer=InMemorySaver(), context_schema=RepContext)
REP = RepContext(rep_id="rep-3", rep_name="Jane Peacock")
TICKET_14 = ("Ticket 14: the customer was charged twice for The Essential Miles Davis on 2013-11-02. "
             "Invoices 9001 and 9002. Please sort it out.")
TICKET_22 = ("Ticket 22: customer 16's box set on invoice 9003 arrived with one of its four discs cracked. "
             "Please file a refund.")

def decide(thread_id, text, decision):
    # Run one ticket to the pause on its own thread and answer with one decision; return the final run.
    cfg = {"configurable": {"thread_id": thread_id}}
    paused = agent.invoke({"messages": [("user", text)]}, cfg, context=REP, version="v2")
    if not paused.interrupts:            # the model answered without proposing a refund
        raise RuntimeError(f"{thread_id}: no pause. Reply: {paused.value['messages'][-1].text[:200]}")
    print(thread_id, "proposed:", paused.interrupts[0].value["action_requests"][0]["args"])
    return agent.invoke(Command(resume={"decisions": [decision]}), cfg, context=REP, version="v2")

## 1. A key for the refund, not the conversation

With the lab's key (thread, invoice, amount), two conversations about ticket 14:

In [5]:
for thread_id in ("monday", "tuesday"):
    done = decide(thread_id, TICKET_14, {"type": "approve"})
    print("   tool:", done.value["messages"][-2].text)
show_rows()

monday proposed: {'invoice_id': 9002, 'amount': 9.99, 'reason': 'duplicate_charge'}


   tool: {"request_id":1,"idempotency_key":"monday:9002:9.99","invoice_id":9002,"amount":9.99,"reason":"duplicate_charge","rep_id":"rep-3","status":"pending_review","created_at":"2026-10-13T12:36:19+00:00","replayed":false}


tuesday proposed: {'invoice_id': 9002, 'amount': 9.99, 'reason': 'duplicate_charge'}


   tool: {"request_id":2,"idempotency_key":"tuesday:9002:9.99","invoice_id":9002,"amount":9.99,"reason":"duplicate_charge","rep_id":"rep-3","status":"pending_review","created_at":"2026-10-13T12:36:24+00:00","replayed":false}
(1, 9002, 9.99, 'pending_review', 'monday:9002:9.99')
(2, 9002, 9.99, 'pending_review', 'tuesday:9002:9.99')


Now a key from the refund alone. The tool looks `idempotency_key` up when it runs, so redefining it is enough.

In [6]:
def idempotency_key(runtime, invoice_id, amount):
    return f"refund:{invoice_id}:{amount:.2f}"

for thread_id in ("wednesday", "thursday"):
    done = decide(thread_id, TICKET_14, {"type": "approve"})
    print("   tool:", done.value["messages"][-2].text)
show_rows()

wednesday proposed: {'invoice_id': 9002, 'amount': 9.99, 'reason': 'duplicate_charge'}


   tool: {"request_id":3,"idempotency_key":"refund:9002:9.99","invoice_id":9002,"amount":9.99,"reason":"duplicate_charge","rep_id":"rep-3","status":"pending_review","created_at":"2026-10-13T12:36:28+00:00","replayed":false}


thursday proposed: {'invoice_id': 9002, 'amount': 9.99, 'reason': 'duplicate_charge'}


   tool: {"request_id":3,"idempotency_key":"refund:9002:9.99","invoice_id":9002,"amount":9.99,"reason":"duplicate_charge","rep_id":"rep-3","status":"pending_review","created_at":"2026-10-13T12:36:28+00:00","replayed":true}
(1, 9002, 9.99, 'pending_review', 'monday:9002:9.99')
(2, 9002, 9.99, 'pending_review', 'tuesday:9002:9.99')
(3, 9002, 9.99, 'pending_review', 'refund:9002:9.99')


Our run: with the lab's key, Monday and Tuesday each filed a request, two rows for one 9.99 duplicate charge (`monday:9002:9.99`, `tuesday:9002:9.99`). With the refund key, Wednesday filed request 3 and Thursday got request 3 back with `replayed: true`: one row for two conversations.

What this key swallows: two separate tickets about two different tracks on invoice 46, each asking for 0.99, make the
same key, `refund:46:0.99`. The second request gets the first one back with `replayed: true`, and the second track is
never refunded, without any error. A key names one operation; "one open request per invoice" or "refunds never above
the invoice total" is a business rule, and it belongs in the service that owns the refunds.

Put the lab's key back for the next tasks:

In [7]:
def idempotency_key(runtime, invoice_id, amount):
    thread_id = runtime.config["configurable"]["thread_id"]
    return f"{thread_id}:{invoice_id}:{amount:.2f}"

## 2. Optional: two refunds in one step

SCRIPTED MODEL · NO API CALL. One AI message with two tool calls: the middleware puts both in one interrupt.

In [8]:
from langchain_core.messages import AIMessage
from shared.scripted import scripted

BOTH = AIMessage(content="", tool_calls=[
    {"name": "file_refund_request", "args": {"invoice_id": 9001, "amount": 9.99, "reason": "duplicate_charge"},
     "id": "call_9001"},
    {"name": "file_refund_request", "args": {"invoice_id": 9002, "amount": 9.99, "reason": "duplicate_charge"},
     "id": "call_9002"}])
two = create_agent(scripted(BOTH, AIMessage(content="(scripted) Done.")), tools=TOOLS, middleware=[approval],
                   checkpointer=InMemorySaver(), context_schema=RepContext)
cfg = {"configurable": {"thread_id": "two-at-once"}}
paused = two.invoke({"messages": [("user", TICKET_14)]}, cfg, context=REP, version="v2")
for action in paused.interrupts[0].value["action_requests"]:
    print(action["name"], action["args"])

file_refund_request {'invoice_id': 9001, 'amount': 9.99, 'reason': 'duplicate_charge'}
file_refund_request {'invoice_id': 9002, 'amount': 9.99, 'reason': 'duplicate_charge'}


In [9]:
decisions = [{"type": "reject", "message": "9001 is the first charge; the policy refunds the later invoice."},
             {"type": "approve"}]
for attempt in ([{"type": "approve"}], decisions):
    try:
        two.invoke(Command(resume={"decisions": attempt}), cfg, context=REP, version="v2")
    except ValueError as error:
        print(len(attempt), "decision(s): ValueError:", error)

1 decision(s): ValueError: Number of human decisions (1) does not match number of hanging tool calls (2).
2 decision(s): ValueError: Number of human decisions (1) does not match number of hanging tool calls (2).


Both attempts fail with the same message about **one** decision: the first answer was saved with the thread when the
step failed, and each later resume replays it. Count before you send, and reset a stuck pause: `update_state(cfg,
None)` writes a new checkpoint at the same step without the saved answer.

In [10]:
waiting = two.get_state(cfg).interrupts[0].value["action_requests"]
print("decisions needed:", len(waiting))
two.update_state(cfg, None)
done = two.invoke(Command(resume={"decisions": decisions}), cfg, context=REP, version="v2")
for m in done.value["messages"]:
    if m.type == "tool":
        print(m.tool_call_id, m.status, "|", m.text[:110])
show_rows()

decisions needed: 2
call_9001 error | User rejected the tool call for `file_refund_request` with reason: 9001 is the first charge; the policy refund
call_9002 success | {"request_id":4,"idempotency_key":"two-at-once:9002:9.99","invoice_id":9002,"amount":9.99,"reason":"duplicate_
(1, 9002, 9.99, 'pending_review', 'monday:9002:9.99')
(2, 9002, 9.99, 'pending_review', 'tuesday:9002:9.99')
(3, 9002, 9.99, 'pending_review', 'refund:9002:9.99')
(4, 9002, 9.99, 'pending_review', 'two-at-once:9002:9.99')


Our run: two action requests in one interrupt. One decision raised the `ValueError`; the two decisions then raised the same error about one decision, the replayed first answer. After `update_state(cfg, None)`, 9001 got a tool message with status `error` and the reason, and 9002 was filed (request 4).

## 3. Optional: reject without a reason

Ticket 22 rejected on three threads with no `message`. First the tool message the model reads, then each reply.

In [11]:
for n in (1, 2, 3):
    done = decide(f"no-reason-{n}", TICKET_22, {"type": "reject"})
    if n == 1:
        print("tool message:", done.value["messages"][-2].status, "|", done.value["messages"][-2].text)
    if done.interrupts:
        print("   proposed again and paused:", done.interrupts[0].value["action_requests"][0]["args"])
    else:
        print("   reply:", done.value["messages"][-1].text)
show_rows()

no-reason-1 proposed: {'invoice_id': 9003, 'amount': 120, 'reason': 'damaged_media'}


tool message: error | User rejected the tool call for `file_refund_request` with id call_R3dippVYiE1tnSx5EeE9O7jO. The tool was not executed. Do not retry this tool call unless the user explicitly requests it.
   reply: The invoice shows that the box set arrived damaged, which qualifies for a refund according to our policies on damaged media. However, I encountered an issue while trying to file the refund request.

Would you like me to try again, or is there anything else you would like to do?


no-reason-2 proposed: {'invoice_id': 9003, 'amount': 120, 'reason': 'damaged_media'}


   reply: The invoice for customer 16 shows that the box set arrived with a cracked disc. According to our policy on damaged media, we can file a refund or offer a replacement. Since it’s a damaged item, I’ll proceed with filing a refund request for the full amount of $120.

Let me know if you'd like me to go ahead and file the refund request.


no-reason-3 proposed: {'invoice_id': 9003, 'amount': 120, 'reason': 'damaged_media'}


   reply: I found that the box set on invoice 9003 arrived with one of its discs cracked. According to our policy on damaged media, a refund can be filed since this title has been damaged.

Would you like me to proceed with the refund request?
(1, 9002, 9.99, 'pending_review', 'monday:9002:9.99')
(2, 9002, 9.99, 'pending_review', 'tuesday:9002:9.99')
(3, 9002, 9.99, 'pending_review', 'refund:9002:9.99')
(4, 9002, 9.99, 'pending_review', 'two-at-once:9002:9.99')


Our run: the model read the middleware's default text, *User rejected the tool call ... The tool was not executed. Do not retry this tool call unless the user explicitly requests it.* None of the three replies said that a reviewer had refused. Run 1 took the rejection for a failure (*I encountered an issue while trying to file the refund request*) and asked whether to try again; run 2 said it would file the full 120.00 and asked the rep to confirm; run 3 asked whether to go ahead with the refund. Nothing was filed. With Part D's reason, the reply told the rep to send a replacement disc. Write a rejection for the model: what happened, and what to do next.

In [12]:
if server is not None:
    server.should_exit = True